In [1]:
import pandas as pd
import numpy as np
import re 
from IPython.display import display, HTML

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, hamming_loss, jaccard_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)

data= pd.read_csv("anti_total.csv", index_col=False)

In [3]:
roomno_mapping = {'A': '1', 'C': '2', 'D': '3', 'E': '4', 'H': '5', 'K': '6'}
data['ROOMNO'] = data['ROOMNO'].map(roomno_mapping)          

data['SEX'] = data['SEX'].map({'M': 1, 'F': 0})

yn_cols = [
    'ISSEPSIS0', 'FEVER', 'DM', 'CARDIOVASCULAR', 
    'RESPIRATORY', 'CNS', 'CANCER', 'LIVER', 'KIDNEY', 'AUTOIMMUNE'
]

for col in yn_cols:
    data[col] = data[col].map({'Y': 1, 'N': 0})

In [4]:
start_index = data.columns.get_loc('Acyclovir')
end_index = data.columns.get_loc('tenofovir/emtricitabine/rilpivirine')

In [5]:
abx_cols = data.columns[start_index:end_index+1]
col_sum = data[abx_cols].sum()

In [6]:
final_cols = col_sum[col_sum >= 500].index.tolist()
base_cols = [c for c in data.columns if c not in abx_cols]
data_filter = data[base_cols + final_cols]

In [7]:
feature_cols = list(set(data_filter.columns) - set(abx_cols))
X = data_filter[feature_cols]
y = data_filter[final_cols]

In [8]:
X = X.drop(columns=['ACCOUNTNO','ROOMNO','AUTIBIOTIC_GROUP'])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 123)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((22374, 39), (22374, 20), (5594, 39), (5594, 20))

In [10]:
X_train.isnull().sum(), X_test.isnull().sum()

(VITALSIGNSRR                 863
 CHECKITEM28A               18126
 SEX                            0
 STAYTIME                       0
 CHECKITEM29SCORE           17507
 Lymphocyte                 15194
 VITALSIGNSPR                 904
 DM                             0
 VITALSIGNSGCS                333
 CHECKITEM27                18407
 INFECTIONSITE2                 0
 INFECTIONSITE4                 0
 VITALSIGNSBT                 245
 OTHERINFECTIONSITE_flag        0
 CARDIOVASCULAR                 0
 INFECTIONSITE3                 0
 LIVER                          0
 CHECKITEM31SCORE           21306
 VITALSIGNSDBP               2525
 CHECKITEM30SCORE           17670
 CNS                            0
 INFECTIONSITE1                 0
 INFECTIONSITE9                 0
 RESPIRATORY                    0
 AUTOIMMUNE                     0
 KIDNEY                         0
 CRP                        16651
 FEVER                          0
 CHECKITEM32SCORE           18465
 CANCER       

In [11]:
X_train.dtypes, X_test.dtypes

(VITALSIGNSRR               float64
 CHECKITEM28A               float64
 SEX                          int64
 STAYTIME                   float64
 CHECKITEM29SCORE           float64
 Lymphocyte                 float64
 VITALSIGNSPR               float64
 DM                           int64
 VITALSIGNSGCS              float64
 CHECKITEM27                float64
 INFECTIONSITE2               int64
 INFECTIONSITE4               int64
 VITALSIGNSBT               float64
 OTHERINFECTIONSITE_flag      int64
 CARDIOVASCULAR               int64
 INFECTIONSITE3               int64
 LIVER                        int64
 CHECKITEM31SCORE           float64
 VITALSIGNSDBP              float64
 CHECKITEM30SCORE           float64
 CNS                          int64
 INFECTIONSITE1               int64
 INFECTIONSITE9               int64
 RESPIRATORY                  int64
 AUTOIMMUNE                   int64
 KIDNEY                       int64
 CRP                        float64
 FEVER                      

In [12]:
y_train.sum().sort_values(ascending=False)

Amoxicillin/Clavulanic acid    4915.0
Flomoxef                       4427.0
Cefixime                       3063.0
Cefazolin                      2192.0
Ciprofloxacin                  2158.0
Cefuroxime                     2078.0
Piperacillin/Tazobactam        2056.0
Azithromycin                   2046.0
Cefoperazone/sulbactam         1236.0
Metronidazole                  1222.0
Levofloxacin                   1145.0
Baloxavir marboxil              932.0
Peramivir                       927.0
Cefadroxil                      895.0
Clindamycin                     796.0
Oseltamivir                     682.0
Ceftriaxone                     663.0
Gentamicin                      653.0
Cephalexin                      515.0
Doxycycline                     422.0
dtype: float64

In [15]:
# 轉數值
num_cols = ['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP'
           , 'WBC', 'Lymphocyte', 'CRP']

for col in num_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

In [16]:
# vital sign impute
vital_cols = ['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP'
             , 'WBC', 'Lymphocyte', 'CRP']

for col in vital_cols:
    # X_train[col + "_missing"] = X_train[col].isna().astype(int) # missing indicator
    median = X_train[col].median()
    X_train[col] = X_train[col].fillna(median)
    
    # X_test[col + "_missing"] = X_test[col].isna().astype(int) # missing indicator
    X_test[col] = X_test[col].fillna(median)

In [17]:
X_train.columns

Index(['VITALSIGNSRR', 'CHECKITEM28A', 'SEX', 'STAYTIME', 'CHECKITEM29SCORE',
       'Lymphocyte', 'VITALSIGNSPR', 'DM', 'VITALSIGNSGCS', 'CHECKITEM27',
       'INFECTIONSITE2', 'INFECTIONSITE4', 'VITALSIGNSBT',
       'OTHERINFECTIONSITE_flag', 'CARDIOVASCULAR', 'INFECTIONSITE3', 'LIVER',
       'CHECKITEM31SCORE', 'VITALSIGNSDBP', 'CHECKITEM30SCORE', 'CNS',
       'INFECTIONSITE1', 'INFECTIONSITE9', 'RESPIRATORY', 'AUTOIMMUNE',
       'KIDNEY', 'CRP', 'FEVER', 'CHECKITEM32SCORE', 'CANCER',
       'CHECKITEM28SCORE', 'ISSEPSIS0', 'WBC', 'MAP', 'AGE', 'VITALSIGNSSPO2',
       'INFECTIONSITE5', 'INJURELEVEL', 'CHECKITEM27SCORE'],
      dtype='object')

In [18]:
drop_cols = ['CHECKITEM28A', 'CHECKITEM27']
X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

In [19]:
# fill score
score_cols = ['CHECKITEM27SCORE', 'CHECKITEM28SCORE', 'CHECKITEM29SCORE', 'CHECKITEM30SCORE', 'CHECKITEM31SCORE', 'CHECKITEM32SCORE']

for col in score_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_train[col] = X_train[col].fillna(-1)

    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
    X_test[col] = X_test[col].fillna(-1)

In [20]:
X_train

,VITALSIGNSRR,SEX,STAYTIME,CHECKITEM29SCORE,Lymphocyte,VITALSIGNSPR,DM,VITALSIGNSGCS,INFECTIONSITE2,INFECTIONSITE4,VITALSIGNSBT,OTHERINFECTIONSITE_flag,CARDIOVASCULAR,INFECTIONSITE3,LIVER,CHECKITEM31SCORE,VITALSIGNSDBP,CHECKITEM30SCORE,CNS,INFECTIONSITE1,INFECTIONSITE9,RESPIRATORY,AUTOIMMUNE,KIDNEY,CRP,FEVER,CHECKITEM32SCORE,CANCER,CHECKITEM28SCORE,ISSEPSIS0,WBC,MAP,AGE,VITALSIGNSSPO2,INFECTIONSITE5,INJURELEVEL,CHECKITEM27SCORE
26479,18.0,1,803.0,0.0,16.0,138.0,0,15.0,0,0,37.8,0,0,0,0,-1.0,87.0,0.0,1,1,0,1,1,0,2.884,0,1.0,1,0.0,1,3.09,103.0,48.0,94.0,0,3.0,2.0
11531,15.0,1,118.0,-1.0,14.6,97.0,0,15.0,0,0,37.3,0,0,0,0,-1.0,72.0,-1.0,0,0,0,0,0,0,2.884,1,-1.0,0,-1.0,0,10.17,87.0,16.0,99.0,0,3.0,-1.0
15854,20.0,1,39.0,-1.0,48.3,55.0,0,15.0,0,0,35.8,0,1,0,0,-1.0,88.0,-1.0,0,0,0,0,0,0,2.884,1,-1.0,0,-1.0,0,5.05,106.0,65.0,99.0,0,2.0,-1.0
26104,18.0,1,135.0,-1.0,12.6,72.0,0,15.0,1,0,36.7,0,0,0,0,-1.0,86.0,-1.0,0,0,0,0,0,0,2.913,1,-1.0,0,-1.0,0,9.17,102.0,24.0,100.0,0,3.0,-1.0
22286,20.0,0,1715.0,-1.0,11.0,66.0,0,15.0,0,0,36.9,0,0,0,0,-1.0,50.0,-1.0,0,1,0,0,0,0,21.035,1,-1.0,0,-1.0,0,9.60,60.0,62.0,97.0,0,2.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15377,18.0,0,172.0,-1.0,3.3,108.0,0,15.0,0,0,37.5,0,0,0,0,-1.0,65.0,-1.0,0,0,0,0,0,0,3.733,1,-1.0,0,-1.0,0,12.82,81.0,39.0,97.0,0,3.0,-1.0
21602,18.0,1,647.0,0.0,8.8,109.0,0,15.0,1,0,37.1,0,0,0,0,2.0,51.0,0.0,1,1,0,0,1,0,0.290,1,0.0,0,0.0,1,8.43,62.0,59.0,100.0,0,4.0,2.0
17730,20.0,1,19.0,-1.0,37.6,37.0,0,15.0,0,0,36.1,0,0,0,0,-1.0,86.0,-1.0,0,0,0,0,0,0,2.884,1,-1.0,0,-1.0,0,4.57,126.0,85.0,100.0,0,2.0,-1.0
15725,18.0,0,2572.0,0.0,17.6,130.0,1,15.0,1,0,37.0,0,1,1,0,-1.0,49.0,0.0,1,1,0,0,1,0,5.444,1,1.0,1,0.0,1,14.52,58.0,77.0,99.0,0,2.0,0.0


In [21]:
X_train.isna().sum()                      

VITALSIGNSRR               0
SEX                        0
STAYTIME                   0
CHECKITEM29SCORE           0
Lymphocyte                 0
VITALSIGNSPR               0
DM                         0
VITALSIGNSGCS              0
INFECTIONSITE2             0
INFECTIONSITE4             0
VITALSIGNSBT               0
OTHERINFECTIONSITE_flag    0
CARDIOVASCULAR             0
INFECTIONSITE3             0
LIVER                      0
CHECKITEM31SCORE           0
VITALSIGNSDBP              0
CHECKITEM30SCORE           0
CNS                        0
INFECTIONSITE1             0
INFECTIONSITE9             0
RESPIRATORY                0
AUTOIMMUNE                 0
KIDNEY                     0
CRP                        0
FEVER                      0
CHECKITEM32SCORE           0
CANCER                     0
CHECKITEM28SCORE           0
ISSEPSIS0                  0
WBC                        0
MAP                        0
AGE                        0
VITALSIGNSSPO2             0
INFECTIONSITE5

In [22]:
y_train.sum(axis=1).mean() # 每人平均用1.65個抗生素

1.4759542325914008

In [23]:
scaled_cols = ['STAYTIME', 'VITALSIGNSGCS', 'VITALSIGNSBT', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'AGE', 
               'VITALSIGNSRR', 'MAP', 'VITALSIGNSPR']

scaler = StandardScaler()

X_train[scaled_cols] = scaler.fit_transform(X_train[scaled_cols])
X_test[scaled_cols] = scaler.fit_transform(X_test[scaled_cols])

In [24]:
RandomForestClassifier?

Init signature:
RandomForestClassifier(
    n_estimators=100,
    *,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features='sqrt',
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=False,
    n_jobs=None,
    random_state=None,
    verbose=0,
    warm_start=False,
    class_weight=None,
    ccp_alpha=0.0,
    max_samples=None,
    monotonic_cst=None,
)
Docstring:     
A random forest classifier.

A random forest is a meta estimator that fits a number of decision tree
classifiers on various sub-samples of the dataset and uses averaging to
improve the predictive accuracy and control over-fitting.
Trees in the forest use the best split strategy, i.e. equivalent to passing
`splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeClassifier`.
The sub-sample size is controlled with the `max_samples` parameter if
`bootstrap=True` (default), otherwise the

In [25]:
base_model = RandomForestClassifier(
                                    n_estimators=500, 
                                    class_weight='balanced',
                                    min_samples_leaf=5,
                                    min_samples_split=10,
                                    max_depth=15,
                                    n_jobs=-1,
                                    random_state=123)
multi_model = MultiOutputClassifier(base_model)

multi_model.fit(X_train, y_train)

,estimator,RandomForestC...dom_state=123)
,n_jobs,None
,n_estimators,500
,criterion,'gini'
,max_depth,15
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0


In [26]:
y_pred_train = multi_model.predict(X_train)
print(f1_score(y_train, y_pred_train, average='micro'))
print(f1_score(y_train, y_pred_train, average='macro'))

0.6159940071216738
0.6097752413320863


In [28]:
y_pred = multi_model.predict(X_test)
print(y_pred[:5])

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [34]:
y_prob_list = multi_model.predict_proba(X_test)

# y_prob = np.array([prob[:,1] for prob in y_prob_list]).T
# print(y_prob[0])

In [22]:
# proba_matrix = np.column_stack([p[:,1] for p in y_prob_list])
# y_pred = (proba_matrix > 0.2).astype(int)

整體的 Top-3 命中率: 67.68%


In [29]:
print(f1_score(y_test, y_pred, average='micro'))
print(f1_score(y_test, y_pred, average='macro'))

0.4002375296912114
0.36021839698477814


In [30]:
hamming_loss(y_test, y_pred)

0.12638541294243832

In [31]:
jaccard_score(y_test, y_pred, average='samples', zero_division=0)

0.2780028630448723

In [35]:
# top k hits rate

def hit_rate_at_k(y_true, proba, k=3):
    
    if isinstance(proba, list):
       proba = np.column_stack([p[:, 1] for p in proba])
    
    topk = np.argsort(proba, axis=1)[:, -k:]
    hits = 0
    for i in range(len(y_true)):
        actual = np.where(y_true[i]==1)[0]
        if len(set(actual)& set(topk[i].tolist())):
            hits += 1
    return hits /  len(y_true)

In [36]:
hit3 = hit_rate_at_k(y_test.values, y_prob_list, k=3)
print('Hit@3: ', hit3)

Hit@3:  0.6961029674651412


In [37]:
# recall@3

def recall_at_k(y_true, proba, k=3):
    if isinstance(proba, list):
        proba = np.column_stack([p[:, 1] for p in proba])

    topk = np.argsort(proba, axis=1)[:, -k:]
    hits = 0
    recalls = []
    for i in range(len(y_true)):
        actual = np.where(y_true[i]==1)[0]
        if len(actual) == 0:
            continue
        recall = len(set(actual) & set(topk[i])) / len(actual)
        recalls.append(recall)
    return np.mean(recalls)

In [38]:
recall_3 = recall_at_k(y_test.values, y_prob_list, k=3)
print('Recall@3: ', recall_3)

Recall@3:  0.6113791260189764


In [39]:
# MAP@3

def map_at_k(y_true, proba, k=3):
    if isinstance(proba, list):
        proba = np.column_stack([p[:, 1] for p in proba])

    topk = np.argsort(proba, axis=1)[:, -k:]
    hits = 0
    APs = []
    for i in range(len(y_true)):
        actual = np.where(y_true[i]==1)[0]
        if len(actual) == 0:
            continue
        score = 0
        hits = 0

        for j in range(k):
            if topk[i][j] in actual:
                hits += 1
                score += hits / (j + 1)
        APs.append(score / min(len(actual), k))
    return np.mean(APs)

In [40]:
map_3 = map_at_k(y_test.values, y_prob_list, k=3)
print('MAP@3: ', map_3)

MAP@3:  0.3541575719779649


In [42]:
target_names = y_train.columns

for i, col in enumerate(target_names):
    print(f'-- {col} --')
    print(classification_report(y_test.iloc[:, i], y_pred[:, i]))

-- Amoxicillin/Clavulanic acid --
              precision    recall  f1-score   support

         0.0       0.87      0.81      0.84      4389
         1.0       0.45      0.57      0.50      1205

    accuracy                           0.76      5594
   macro avg       0.66      0.69      0.67      5594
weighted avg       0.78      0.76      0.77      5594

-- Azithromycin --
              precision    recall  f1-score   support

         0.0       0.94      0.89      0.91      5109
         1.0       0.26      0.42      0.32       485

    accuracy                           0.84      5594
   macro avg       0.60      0.65      0.62      5594
weighted avg       0.88      0.84      0.86      5594

-- Baloxavir marboxil --
              precision    recall  f1-score   support

         0.0       0.98      0.94      0.96      5359
         1.0       0.29      0.53      0.37       235

    accuracy                           0.93      5594
   macro avg       0.63      0.74      0.67      5

In [41]:
multi_model_importance =  multi_model.estimators_[0].feature_importances_
importance_df = pd.DataFrame({'Features': X_train.columns, 'Importance':multi_model_importance})
print(importance_df.sort_values(by='Importance', ascending=False))

                   Features  Importance
2                  STAYTIME    0.126639
34           INFECTIONSITE5    0.114906
8            INFECTIONSITE2    0.093974
32                      AGE    0.091072
5              VITALSIGNSPR    0.050645
10             VITALSIGNSBT    0.046986
31                      MAP    0.043516
16            VITALSIGNSDBP    0.041460
13           INFECTIONSITE3    0.036304
4                Lymphocyte    0.032560
35              INJURELEVEL    0.031996
30                      WBC    0.030977
19           INFECTIONSITE1    0.027476
24                      CRP    0.025861
33           VITALSIGNSSPO2    0.024004
0              VITALSIGNSRR    0.020008
1                       SEX    0.016738
29                ISSEPSIS0    0.015685
7             VITALSIGNSGCS    0.012569
12           CARDIOVASCULAR    0.010237
25                    FEVER    0.010220
3          CHECKITEM29SCORE    0.009839
23                   KIDNEY    0.009713
17         CHECKITEM30SCORE    0.008914
